# Overnight CatBoost — five seeded, reproducible benchmark runs

**Start this using Kaggle → Save Version → Save & Run All (not Quick Save, not the interactive Run All button).** Kaggle version runs start in a fresh session and continue in the cloud after you close the browser. Maximum documented per notebook is **12 hours**; our experiment stops scheduling new seeds after **9 hours**, and completed seeds are archived to GitHub immediately.

**What this does automatically:**
1. Clone [the research GitHub repository](https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids). Recover the original SCV reference split and two previous LightGBM model checkpoints from **SHA-256 pinned private draft GitHub Releases**, because the version run starts with a clean `/kaggle/working`. No retraining of earlier models.
2. Train CatBoost multiclass on exactly **140,328 original training-fold target rows** with the identical **35,005 training-derived validation target rows**, using **five fixed random seeds (42–46)** and two predefined architectures per seed (unweighted and moderately rare-weighted).
3. Evaluate **eight fixed probability ensembles per seed** combining those real CatBoost model outputs with the old frozen LightGBM predictions. For every real model: accuracy, weighted precision, weighted recall, weighted F1, ten-class macro-F1, balanced accuracy, complete confusion and class support.
4. Generate **5 plots × PNG/SVG/PDF = 15 evidence-backed vector and raster files per seed**, and a cross-seed summary that displays actual mean/std once complete.
5. **After each seed finishes, before training another seed:** back up all model weights, probabilities, checkpoints, figures and plot sources to an **unpublished GitHub draft Release**, verify SHA-256, and sync *only aggregate numerical results/figures* to the public GitHub repo. Every completed seed is preserved independently.
6. Keep the designated **UNSW-NB15 official test dataset unopened and completely locked** until final method selection. Current results are not official-test comparisons with SCV-IDS.

**Required notebook settings before Save & Run All:**
- Attach Kaggle dataset `dsshayan/unsw-nb15`.
- Turn **Internet ON**, to clone the repo, install CatBoost if needed, and recover draft archives.
- Authorize the existing Kaggle secret named `pushRare` for **this new notebook**, so private model archives can be restored and saved. Do not enter or print the token.
- CPU is sufficient; **GPU accelerator may remain OFF**.
- Kaggle commit execution limit applies; cloud reliability and remaining quota are not guaranteed. Logs and results can be reviewed tomorrow from the notebook's Versions panel.

**IMPORTANT:** This is a repeated-seed *strong baseline* screen pursuing both SCV-IDS's reported **85.77% multiclass accuracy** and better macro-F1. CatBoost itself is not a novel research method; showing both metrics on an internal split does not establish superiority to a paper evaluated on a different split. We will finalize a novel mechanism and official-test protocol after comparing measured data.

In [ ]:
# ONE CLOUD-RUN CELL — fully self-contained even in fresh Save & Run All state.
# DO NOT run this interactively and then shut down the browser.
# Use Kaggle's Save Version > Save & Run All (background cloud commit).
from pathlib import Path
import hashlib
import importlib.util
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/rare-attack-ids-overnight")
BASE = Path("/kaggle/working/research_outputs")
OUTPUT = Path("/kaggle/working/CATBOOST_OVERNIGHT_20261018")
DATA = Path(
    "/kaggle/input/datasets/dsshayan/unsw-nb15/"
    "UNSW_NB15_training-set.csv"
)
SCV_ARCHIVE = Path("/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
TAB_ARCHIVE = Path("/kaggle/working/CTRL-TAB-001-S42_FULL_ARTIFACTS.tar")
EXPECTED_TRAIN_SHA = (
    "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
)
URL = "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"

# Fail fast if the dataset or private draft-archive secret is missing.
assert DATA.is_file(), (
    "Attach dsshayan/unsw-nb15 to this Kaggle notebook before Save & Run All"
)
try:
    github_token = UserSecretsClient().get_secret("pushRare")
except Exception as exc:
    raise RuntimeError(
        "Enable and authorize Kaggle secret pushRare for THIS notebook."
    ) from exc
if not github_token:
    raise RuntimeError("The Kaggle pushRare secret is empty")

# Freeze source and reproduce all original reference targets exactly.
with DATA.open("rb") as f:
    train_sha = hashlib.file_digest(f, "sha256").hexdigest()
assert train_sha == EXPECTED_TRAIN_SHA, (
    f"Dataset integrity mismatch: {train_sha}"
)
print("UNSW TRAINING DATASET SHA-256 VERIFIED", flush=True)
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", URL, str(REPO)],
        check=True
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("FROZEN RESEARCH CODE COMMIT:", commit, flush=True)

if importlib.util.find_spec("catboost") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "catboost>=1.2.8,<2"],
        check=True
    )

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src")
env["GITHUB_TOKEN"] = github_token
env["OMP_NUM_THREADS"] = "4"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MPLBACKEND"] = "Agg"

# Clean Kaggle version sessions DO NOT inherit interactive model files.
# Restore both archives and verify every source file cryptographically.
subprocess.run([
    sys.executable, str(REPO / "scripts/restore_scv_release.py"),
    "--base", str(BASE),
    "--archive", str(SCV_ARCHIVE)
], cwd=str(REPO), env=env, check=True)

subprocess.run([
    sys.executable, str(REPO / "scripts/restore_ctrl003_tabular.py"),
    "--base", str(BASE),
    "--archive", str(TAB_ARCHIVE)
], cwd=str(REPO), env=env, check=True)

print("BOTH FROZEN BASELINES RESTORED WITHOUT RETRAINING", flush=True)
print("STARTING FIVE SEEDS — EACH SEED ARCHIVED INDEPENDENTLY", flush=True)

try:
    subprocess.run([
        sys.executable, str(REPO / "scripts/run_overnight_catboost.py"),
        "--repo", str(REPO),
        "--reference", str(BASE / "SCV-IDS-REPRO-001"),
        "--baseline", str(BASE / "CTRL-TAB-001-S42"),
        "--config", str(REPO / "configs/ctrl004_catboost_pareto_seed42.json"),
        "--output", str(OUTPUT),
        "--max-hours", "9.0"
    ], cwd=str(REPO), env=env, check=True)
finally:
    env.pop("GITHUB_TOKEN", None)
    del github_token

print("OVERNIGHT JOB FINISHED — ALL COMPLETED SEEDS BACKED UP", flush=True)
print("Look in Kaggle Notebook Version Outputs:", OUTPUT, flush=True)


## Tomorrow morning

Open the Kaggle notebook **Version** you saved tonight, read the end of the execution log and the `OVERNIGHT_STATUS.json`, `ALL_SEED_SCORES.csv`, and `ACROSS_SEED_SUMMARY.csv` outputs. You can send us just the text under **`OVERNIGHT CATBOOST EXPERIMENTS COMPLETED AND ARCHIVED`** or the cross-seed summary table. There is no requirement to keep your computer on.

If an individual seed fails, finished seeds have already been archived separately to an unpublished GitHub draft Release. The Kaggle version may report Failed and may not retain *all* local outputs, but previously uploaded GitHub draft archives remain available. Never claim numerical superiority over the base paper's designated test based only on this internal-development holdout.